In [ ]:
# Libraries
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

from scipy.stats import pearsonr, spearmanr
from scipy.stats import skew

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

import importlib
import helpers
importlib.reload(helpers)
from helpers import *
import re

: 

In [ ]:
# Was the VISIT_ORDER properly imported?
print(VISIT_ORDER[:5])
print(type(VISIT_ORDER))

In [ ]:
# data/raw
list_files()

#  Demographic data

- What are the main demographics variables?
- Rename values for clarity and fill in missing values.
- How are the visits encoded? Are they ordered?
- Check the link between VISCODE and VISCODE2 - can we easily translate between the two?

In [ ]:
DATA_PATH = 'data/raw'
os.chdir(DATA_PATH)

In [ ]:
# Registry with visit codes to help create an ordered column of visit codes
reg = pd.read_csv('REGISTRY_16Oct2025.csv')
reg[['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE']].head(10)

In [ ]:
# Find all unique VISCODE and VISCODE2 pairs and create a dictionary
# VISCODE2 will be used because it is informative -> m24 = 24. month
viscode_pairs = reg[['VISCODE', 'VISCODE2']].drop_duplicates()
viscode_pairs = viscode_pairs[viscode_pairs['VISCODE'] != '-4'] # remove missing
print(viscode_pairs)

In [ ]:
# demographics data
dem = pd.read_csv('PTDEMOG_15Oct2025.csv')

In [ ]:
inspect_df(dem)

In [ ]:
print(dem.columns)

In [ ]:
# Keep relevant columns, rename for clarity
dem = dem[['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'PTGENDER',
            'PTDOBYY', 'PTMARRY', 'PTEDUCAT', 'PTNOTRT',
            'PTHOME', 'PTADDX', 'PTETHCAT', 'PTRACCAT']]
dem.columns = ['PHASE', 'RID', 'VISCODE','VISCODE2', 'VISDATE', 'sex',
               'birth_year', 'marital_status', 'years_of_education', 'retired',
               'home', 'year_diagnosed_ad', 'ethnicity', 'race']

# Drop entries with failed screening
print(dem.shape)
dem = dem[dem['VISCODE'] != 'f']
print(dem.shape)

In [ ]:
# -4 in the dataset means NaN
dem.replace(-4, np.nan, inplace=True)
dem.replace(' -4', np.nan, inplace=True)

In [ ]:
dem.isnull().sum()

In [ ]:
# Fill NA and preprocess the dataframe
cols_to_fill = ['sex', 'birth_year', 'marital_status', 'years_of_education',
                'home', 'ethnicity', 'race']

dem = preprocess_adni_df(dem, cols_to_fill=cols_to_fill)

In [ ]:
dem.isna().sum()

In [ ]:
dem['year_diagnosed_ad'].unique()

In [ ]:
dem['year_diagnosed_ad'] = dem['year_diagnosed_ad'].replace(9999, np.nan)

In [ ]:
dem[dem['sex'].isna()].head()

In [ ]:
dem[dem['birth_year'].isna()]

In [ ]:
# any non-null value exists for birth_year within each RID group
df = dem[dem['birth_year'].isna()]
df_grouped = (
    df.groupby('RID')['birth_year']
    .apply(lambda x: x.notna().any())
    .reset_index(name='dob_exists')
    )

df_grouped

In [ ]:
# If we check for each RID above, we see that these participants only visited for screening. 
# For those, without sex and yob data, and with no other entries to fill in the missing data, we remove them.
dem = dem.dropna(subset=['birth_year'])

In [ ]:
dem[dem['marital_status'].isna()]

In [ ]:
dem.isnull().sum()

In some cases, missing entries can be deducted from previous visits or based on the entries in onther files. Since the cases are individual anad difficult to program into a function, but the retention of data on this small sample is important, I'll do it manually case by case. I mostly consult the Registry ('REGISTRY_16Oct2025.csv')

In [ ]:
dem[dem['VISCODE2'].isna()]

In [ ]:
dem[dem['RID'] == 6970]

In [ ]:
dem[dem['RID'] == 7112]

In [ ]:
dem[dem['RID'] == 10417]

In [ ]:
mask = dem['VISCODE'].str.contains('init', na=False) & dem['VISCODE2'].isna()
dem.loc[mask, 'VISCODE2'] = 'bl'

In [ ]:
dem[dem['VISCODE2'].isna()]

In [ ]:
dem.loc[dem['VISCODE2'].isna(), 'VISCODE2'] = 'sc'

In [ ]:
# Find all RIDs with at least one missing value in 'race' column
rids_with_missing_race = dem[dem['race'].isna()]['RID'].unique()

# Filter the original dataframe to include all rows corresponding to these RIDs
df_filtered = dem[dem['RID'].isin(rids_with_missing_race)]
df_filtered

In [ ]:
dem[dem['marital_status'].isna()]

In [ ]:
dem[dem['home'].isna()]

In [ ]:
dem = dem.sort_values(['RID', 'VISDATE'])
dem['VISCODE2'] = dem.groupby('RID')['VISCODE2'].transform(lambda x: x.ffill().bfill())

In [ ]:
dem.isnull().sum()

In [ ]:
dem[dem['VISDATE'].isna()]

In [ ]:
dem[dem['RID'] == 4737]

In [ ]:
reg[reg['RID'] == 4737][['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE']]

In [ ]:
dem[dem['RID'] == 2238]

In [ ]:
dem.loc[(dem['RID'] == 4737) & (dem['VISCODE'] == 'v11'), 'VISDATE'] = pd.Timestamp('2013-05-07')

In [ ]:
dem[dem['RID'] == 6617]

In [ ]:
reg[reg['RID'] == 6617][['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE']]

In [ ]:
reg[reg['RID'] == 10417][['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE']]

In [ ]:
dem.loc[(dem['RID'] == 10417) & (dem['VISCODE2'].isna()), 'VISCODE2'] = 'sc'

In [ ]:
dem[dem['home'].isna()]

In [ ]:
# Check for duplicated entries, participants who maybe switched diagnosis and continue the study
# How many eligible unique participants included for more phases?
duplicates = dem['RID'].duplicated().sum()
if duplicates > 0:
  duplicate_ids = dem[dem['RID'].duplicated(keep=False)]['RID'].unique()
  duplicate_counts = dem[dem['RID'].isin(duplicate_ids)].groupby('RID')['VISCODE'].count().reset_index()
  duplicate_counts.columns = ['RID', 'visit_count']
  duplicate_counts = duplicate_counts.sort_values('visit_count', ascending=False)

  print(f"No. of unique participants: {dem['RID'].nunique()}")
  print(f"\nParticipants with repeated visits ({duplicates} duplicates found):")
  print(duplicate_counts.to_string(index=False))

In [ ]:
sns.countplot(x='visit_count', data=duplicate_counts)
plt.title('Distribution of Number of Visits per Participant')
plt.xlabel(f"Number of Visits ({dem.shape[0]} total)")
plt.ylabel(f"Number of Participants ({dem['RID'].nunique()} individuals)")
plt.grid(axis='y', alpha=0.7)
plt.show()

Demographics table might not be the best overview of the visits. Repeat the analysis with the combined dataset, focus only on the gut metabolite cohort.

In [ ]:
dem['race'].unique()

In [ ]:
# Check the unique values in each column before mapping
# Important: no NA in 'sex' and 'year_of_birth'
print(dem['sex'].unique())
print(dem['marital_status'].unique())
print(dem['retired'].unique())
print(dem['home'].unique())
print(dem['ethnicity'].unique())
print(dem['race'].unique())
print(dem['years_of_education'].unique())
print(dem['year_diagnosed_ad'].unique())
print(dem['birth_year'].unique())

In [ ]:
# Recode variable names for clarity using the data dictionary
# at https://adni.loni.usc.edu/data-samples/data-dictionary-search
dem['sex'] = dem['sex'].map({1: 'Male', 2: 'Female', -4: 'Unknown'})
dem['marital_status'] = dem['marital_status'].map({
    1.0: 'Married',
    2.0: 'Widowed',
    3.0: 'Divorced',
    4.0: 'Never Married',
    5.0: 'Unknown',
    6.0: 'Partnership',
    -4.0:'Unknown'
}).fillna('Unknown')

dem['retired'] = dem['retired'].map({
    1.0: 'Yes', 
    0.0: 'No',
    2.0: 'Not applicable',
    -4.0: 'Unknown'
}).fillna('Unknown')

dem['home'] = dem['home'].map({
    1.0: 'House',
    2.0: 'Condo (owned)',
    3.0: 'Apartment (rented)',
    4.0: 'Mobile Home',
    5.0: 'Retirement Community',
    6.0: 'Assisted Living',
    7.0: 'Skilled Nursing Facility',
    8.0: 'Other',
    9.0: 'House',
    10.0: 'House',
    -4.0: 'Unknown'
}).fillna('Unknown')

dem['ethnicity'] = dem['ethnicity'].map({
    1.0:'Hispanic/Latino',
    2.0: 'Not Hispanic',
    3.0: 'Unknown',
    -4.0: 'Unknown'
}).fillna('Unknown')

dem['race_name'] = dem['race'].map({
    '1': 'American Indian or Alaskan Native',
    '2': 'Asian',
    '3': 'Hawaiian or Other Pacific Islander',
    '4': 'Black or African American',
    '5': 'White',
    '6': 'More than one race',
    '7': 'Unknown',
    '8': 'Native Hawaiian',
    '9': 'Other Pacific Islander',
    '-4': 'Unknown',
    '1|5': 'More than one race',
    '1|4': 'More than one race',
    '4|5': 'More than one race',
    '2|5': 'More than one race',
    '3|4|5': 'More than one race',
    '1|7': 'More than one race',
    '1|4|5': 'More than one race',
    '2|4': 'More than one race',
    '5|7': 'More than one race',
    '5|8': 'More than one race'
})

In [ ]:
print(dem.shape)
dem.isnull().sum()

In [ ]:
dem[dem['race'].isna()]

In [ ]:
print(np.sort(dem['birth_year'].unique()))

In [ ]:
# print(dem.dtypes)
# Fix data types
dem['VISDATE'] = pd.to_datetime(dem['VISDATE'])
dem['birth_year'] = dem['birth_year'].astype(int)

In [ ]:
dem.head()

In [ ]:
dem.isnull().sum()

In [ ]:
# dem[dem['RID'] == 2238]

In [ ]:
print(dem['race_name'].unique())
dem['race_name'] = dem['race_name'].fillna('Unknown')

In [ ]:
dem.dtypes

In [ ]:
# Fill NAs by looking up the value for each RID
dem['year_diagnosed_ad'] = dem.groupby('RID')['year_diagnosed_ad'].transform('first')

In [ ]:
# Check if any RIDs still have inconsistent nulls
inconsistent = dem.groupby('RID')['year_diagnosed_ad'].apply(lambda x: x.isnull().any() and x.notnull().any()).sum()
print(f"Number of RIDs with inconsistent data: {inconsistent}")

In [ ]:
dem.isnull().sum()

In [ ]:
# Syntax: df.loc[row_indexer, column_indexer]
dem.loc[dem['year_diagnosed_ad'].isna(), ['VISCODE2', 'RID', 'year_diagnosed_ad']].head()

# Diagnosis

The Diagnosis dataset contains longitudinal clinical diagnosis labels (NL, MCI, AD) assigned at each ADNI visit using cognitive tests, imaging, and clinical judgement. This is the primary outcome variable, defining which participants convert from healthy or mildly impaired to Alzheimer's disease over the course of the study.

In [ ]:
dx = pd.read_csv('DXSUM_16Oct2025.csv')
inspect_df(dx)

In [ ]:
data_dict = pd.read_csv("../metadata/DATADIC_28Oct2025.csv")

In [ ]:
data_dict.head()

In [ ]:
data_dict[data_dict['TBLNAME']=='DXSUM'].head(40)

'DXCONV' and 'DXCONTY' are supposed to convey, which participants converted and to which diagnosis, but they are not in the dataset, hence I will find out the conversion and its type manually.

In [ ]:
dx.columns

In [ ]:
dx = dx[['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'DIAGNOSIS']]
dx.columns = ['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'diagnosis']

In [ ]:
# Are visits already ordered?
print(dx['VISCODE2'].unique())
dx['VISCODE2'].is_monotonic_increasing

In [ ]:
# Are there many unscheduled exams?
dx[dx['VISCODE2'] == 'uns1']

In [ ]:
# We will count it to a m12, even if it is a m08 visit
print(dx[dx['RID'] ==  168])
dx.loc[(dx['RID'] == 168) & (dx['VISCODE2'] == 'uns1'), 'VISCODE2'] = 'm12'

In [ ]:
dx[dx['EXAMDATE'].isna()]

In [ ]:
dx[dx['RID'] == 2238]

In [ ]:
(pd.Timestamp('2025-03-18') - pd.Timestamp('2019-03-06')).days / 30.44

In [ ]:
dx.loc[(dx['RID'] == 2238) & (dx['VISCODE'] == '4_init'), 'EXAMDATE'] = pd.Timestamp('2025-05-01')
dx.loc[(dx['RID'] == 2238) & (dx['VISCODE'] == 'y2'), 'VISCODE2'] = 'm168'


In [ ]:
dx = preprocess_adni_df(dx, cols_to_fill=['diagnosis'])
dx.isnull().sum()

In [ ]:
# Order VISCODE2 as categorical
def generate_vis_codes():
    # Create the list with 'm' followed by multiples of 6 from 6 to 240
    visits_6m = [f"m{str(i).zfill(2)}" for i in range(6, 241, 6)]
    return visits_6m

visits_6m = generate_vis_codes()

print(visits_6m)

visit_order = ['sc', 'bl', 'scmri', 'm03'] + visits_6m

# Order VISCODE2 column
dx['VISCODE2'] = pd.Categorical(dx['VISCODE2'], categories=visit_order, ordered=True)

In [ ]:
dx[dx['VISCODE2'].isna()]

In [ ]:
# # Performed for all RIDs above to check for missing viscodes2
dx[dx['RID'] == 10005]

# # Find VISCODE2 value when VISCODE and PHASE have certain value
# dx[(dx['VISCODE'] == '4_init') & (dx['PHASE'] == 'ADNI4')]

In [ ]:
# Manually fix missing VISCODE2 by checking other data per RID
dx.loc[(dx['RID'] == 6082) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm12'
dx.loc[(dx['RID'] == 2068) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm96'
dx.loc[(dx['RID'] == 5273) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm60'
dx.loc[(dx['RID'] == 4100) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm126'
dx.loc[(dx['RID'] == 6514) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm48'
dx.loc[(dx['RID'] == 4420) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm36'
dx.loc[(dx['RID'] == 1300) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm132'
dx.loc[(dx['RID'] == 7112) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm12'
dx.loc[(dx['RID'] == 6970) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm48'
dx.loc[(dx['RID'] == 10417) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm36'
dx.loc[(dx['RID'] == 10529) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'bl'
dx.loc[(dx['RID'] == 10175) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm12'
dx.loc[(dx['RID'] == 10005) & (dx['VISCODE2'].isna()), 'VISCODE2'] = 'm24'

In [ ]:
dx[dx['VISCODE2'].isna()]

In [ ]:
for col in dx.columns:
    print(f"{col}:", dx[col].unique())

In [ ]:
dx.isnull().sum()

In [ ]:
dx.head(10)

In [ ]:
dx['diagnosis'].dtype

In [ ]:
dx.isnull().sum()

In [ ]:
dx[dx['EXAMDATE'].isna()]

In [ ]:
dx[dx['EXAMDATE'].dt.year >= 2026]

In [ ]:
dx['EXAMDATE'].dtype

In [ ]:
dx = deduce_missing_exam_date(dx)

In [ ]:
dx.shape

In [ ]:
dx['RID'].nunique()

In [ ]:
dx.isnull().sum()

In [ ]:
dx.to_pickle('../interim/clean_diagnosis.pkl')

In [ ]:
dem.head()

In [ ]:
# dem[dem['VISDATE'].dt.year >= 2025].head()

In [ ]:
dem.columns

In [ ]:
dx.head()

In [ ]:
dem_dx = dx.copy()
dem_dx = pd.merge(
    dx[['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'diagnosis']],
    dem[['PHASE', 'RID', 'VISCODE', 'sex', 'birth_year', 'marital_status',
        'years_of_education', 'retired', 'home',
        'year_diagnosed_ad', 'ethnicity', 'race', 'race_name']],
    on=['RID', 'PHASE', 'VISCODE'],
    how ='left'
)

dem_dx.head(20)

In [ ]:
dem_dx.shape

In [ ]:
dem_dx[dem_dx['EXAMDATE'].dt.year >= 2026]

In [ ]:
dem_dx.isnull().sum()

In [ ]:
dem_dx['VISCODE2'] = pd.Categorical(
    dem_dx['VISCODE2'], 
    categories=visit_order, 
    ordered=True
)

In [ ]:
dem_dx.dtypes

In [ ]:
# Some variable values never change
cols_to_fill = ['sex', 'birth_year', 'marital_status', 'years_of_education', 'retired', 'home', 'ethnicity', 'race', 'race_name']
dem_dx = fill_missing_with_known(dem_dx, cols_to_fill, id_col='RID', 
                                   date_col='VISCODE')

In [ ]:
dem_dx.isnull().sum()

In [ ]:
dem_dx['RID'].nunique()

In [ ]:
dem_dx[dem_dx['sex'].isna()]['RID'].nunique()

The demographic table records demographic details (like sex, race, and education) only at the Screening (sc) or Baseline (bl) visit, leaving them blank for follow-up visits.
Since these variables don't change, we use the firsrt available value to fill in NAs.

In [ ]:
# List of columns that are missing demographic data
dem_cols = ['sex', 'birth_year', 'marital_status', 'years_of_education', 
            'retired', 'home', 'ethnicity', 'race', 'race_name']

for col in dem_cols:
    # 1. Create a lookup: Get the first non-null value available for each RID in the dem table
    lookup = dem.groupby('RID')[col].first()
    
    # 2. Fill NaNs in dem_dx by mapping the RID to our lookup values
    dem_dx[col] = dem_dx[col].fillna(dem_dx['RID'].map(lookup))

# Check the results
print(dem_dx.isna().sum())

In [ ]:
dem_dx[dem_dx['birth_year'].isna()]

In [ ]:
# These people only di screening, so we dont have diagnosis, nor continuous data. Drop.
dem_dx.dropna(subset=['birth_year'], inplace=True)

In [ ]:
dem_dx.head()

In [ ]:
dem_dx.isnull().sum()

In [ ]:
dem_dx[dem_dx['diagnosis'].isna()]

In [ ]:
dem_dx.dtypes

In [ ]:
dem_dx['sex'] = dem_dx['sex'].astype('category')
dem_dx['birth_year'] = dem_dx['birth_year'].astype(int)

In [ ]:
dem_dx[dem_dx['EXAMDATE'].isna()]

In [ ]:
dem_dx[dem_dx['RID'] == 7092]

In [ ]:
# Estimate the examdate based on VISCODE2 logic
dem_dx_cleaned = dem_dx.copy().groupby('RID', group_keys=False).apply(fill_exam_dates)

In [ ]:
dem_dx_cleaned['EXAMDATE'].isnull().sum()

In [ ]:
# Write clean df to .pkl to preserve dates and categories
dem_dx_cleaned.to_pickle('../interim/clean_demographics_and_diagnosis.pkl')

# Blood Serum Gut Metabolites

The serum gut metabolite dataset is one of the more compelling finds in this project: ADNI collected blood-based metabolomics data as far back as 2005, at a time when the gut-brain axis was not yet a mainstream focus in AD research. The gut microbiome shapes systemic metabolite levels that can cross the blood-brain barrier and influence neuroinflammation, amyloid clearance, and neurotransmitter synthesis. Ideally we would have direct microbiome cultures (stool metagenomics) to characterise bacterial community composition, but serum metabolite profiles might still offer a meaningful, non-invasive window into gut microbial activity and its downstream effects on brain health.

In [ ]:
mtb = pd.read_csv('ADMCGUTMETABOLITESLONG_12_13_21_15Oct2025.csv')
print(list(mtb.columns))
mtb = mtb.drop(columns=['DRAWDATE', 'DRAWTIME', 'VID', 'SAMPLE_ID', 'SAMPLE_ID', 'update_stamp'])
mtb.shape

In [ ]:
mtb.replace(-4, np.nan, inplace=True)
mtb.isnull().sum()

In [ ]:
inspect_df(mtb)

In [ ]:
mtb[mtb['UDCA'].isna()]

In [ ]:
mtb[mtb['RID'] == 54]

In metabolomics, missing values often occur because the metabolite concentration is below the instrument’s limit of detection (LOD) rather than truly absent. Many papers treat these missing values as very small concentrations instead of random missing data. A widely used approach is: 

$\text{Imputed value} = \frac{\text{minimum detected value}}{2}$

In [ ]:
# impute missing UDCA value
min_val = mtb['UDCA'].min(skipna=True)
mtb['UDCA'] = mtb['UDCA'].fillna(min_val / 2)

In [ ]:
# post-imputation
mtb[mtb['RID'] == 54]

In [ ]:
mtb[mtb['RID'] == 54][['CDCA', 'UDCA']] # CDCA can be converted into UDCA by intestinal bacteria.

The imputed value is plausible, as UDCA levels are usually lower as the CDCA levels (secondary bile acid is made from the primary bile acid by bacteria and is thus lower.)

In [ ]:
mtb.columns.to_list()

In [ ]:
visit_order[:10]

In [ ]:
mtb['VISCODE2'] = pd.Categorical(mtb['VISCODE2'], categories=visit_order, ordered=True)
dx['EXAMDATE'] = pd.to_datetime(dx['EXAMDATE'], errors='coerce')
mtb.dtypes

In [ ]:
dem_dx_cleaned['EXAMDATE'] = pd.to_datetime(dem_dx_cleaned['EXAMDATE'])
mtb['EXAMDATE'] = pd.to_datetime(mtb['EXAMDATE'])
dem_dx_cleaned.isnull().sum()

In [ ]:
dem_dx_cleaned[dem_dx_cleaned['EXAMDATE'].isna()]

In [ ]:
# # Apply to dataset
# dem_dx_cleaned['EXAMDATE'] = dem_dx_cleaned.apply(
#     lambda row: infer_examdate(row, dem_dx_cleaned),
#     axis=1
# )

In [ ]:
dem_subset = dem_dx_cleaned[
    ['EXAMDATE','RID','VISCODE2','diagnosis','sex','birth_year','year_diagnosed_ad']
]

dem_subset.isnull().sum()

In [ ]:
dem_subset = dem_subset.sort_values(['EXAMDATE','RID','VISCODE2'])
mtb = mtb.sort_values(['EXAMDATE','RID','VISCODE2'])

mtb_merged_with_dem_dx = pd.merge_asof(
    mtb,
    dem_subset,
    on='EXAMDATE',
    by=['RID','VISCODE2'],
    direction='nearest',
    tolerance=pd.Timedelta("90D")
)

In [ ]:
print(len(mtb))
print(len(mtb_merged_with_dem_dx))

In [ ]:
mtb_merged_with_dem_dx.isnull().sum()

In [ ]:
cols_to_fill = ['sex', 'birth_year']
mtb_merged_with_dem_dx = fill_missing_with_known(
    mtb_merged_with_dem_dx, 
    cols_to_fill, 
    id_col='RID', 
    date_col='VISCODE'
    )

In [ ]:
partially_missing_rids = (
    mtb_merged_with_dem_dx[mtb_merged_with_dem_dx['diagnosis']
    .isna()]['RID']
    .astype(int)
    .unique()
    .tolist()
)

In [ ]:
mtb_merged_with_dem_dx = fill_partially_missing_values(
    mtb_merged_with_dem_dx, 
    partially_missing_rids
    )

In [ ]:
mtb_merged_with_dem_dx[mtb_merged_with_dem_dx['diagnosis'].isna()].head(20)

In [ ]:
mtb_merged_with_dem_dx.sort_values(by=['RID', 'VISCODE2'])
mtb_merged_with_dem_dx['diagnosis'] = mtb_merged_with_dem_dx.groupby('RID')['diagnosis'].transform(lambda x: x.bfill())

In [ ]:
missing_mtb_data = mtb_merged_with_dem_dx[mtb_merged_with_dem_dx['birth_year'].isna()]

In [ ]:
missing_mtb_data

In [ ]:
# List the columns you want to repair
cols_to_fix = ['sex', 'birth_year']

for col in cols_to_fix:
    # groupby RID and use transform to fill every row with the first valid entry found
    mtb_merged_with_dem_dx[col] = (
        mtb_merged_with_dem_dx.groupby('RID')[col]
        .transform(lambda x: x.ffill().bfill())
    )


In [ ]:
mtb_merged_with_dem_dx[mtb_merged_with_dem_dx['RID'] == 2193]

In [ ]:
# 1. Create a clean lookup table from dem_dx_cleaned
# We drop duplicates to ensure we have one unique row per RID
lookup = dem_dx_cleaned.dropna(subset=['sex', 'birth_year']).drop_duplicates('RID')
lookup_map = lookup.set_index('RID')

# 2. Fill missing 'sex'
mtb_merged_with_dem_dx['sex'] = mtb_merged_with_dem_dx['sex'].fillna(
    mtb_merged_with_dem_dx['RID'].map(lookup_map['sex'])
)

# 3. Fill missing 'birth_year'
mtb_merged_with_dem_dx['birth_year'] = mtb_merged_with_dem_dx['birth_year'].fillna(
    mtb_merged_with_dem_dx['RID'].map(lookup_map['birth_year'])
)

# 4. Final Check
missing_after = mtb_merged_with_dem_dx[['sex', 'birth_year']].isnull().sum()
print("Missing values after update:")
print(missing_after)

In [ ]:
mtb_merged_with_dem_dx.isnull().sum()

In [ ]:
mtb_merged_with_dem_dx[mtb_merged_with_dem_dx['diagnosis'].isnull()]

In [ ]:
dem_dx_cleaned = dem_dx_cleaned.sort_values(['RID', 'EXAMDATE'])

for rid in mtb_merged_with_dem_dx[mtb_merged_with_dem_dx['diagnosis'].isnull()]['RID'].unique():
    print(f"RID: {rid}")
    print(dem_dx_cleaned[dem_dx_cleaned['RID'] == rid][['EXAMDATE', 'VISCODE2', 'diagnosis']])

In [ ]:
# 2193
target_rid = 2193
diagnosis = 2

mtb_merged_with_dem_dx.loc[
    (mtb_merged_with_dem_dx['RID'] == target_rid) & 
    (mtb_merged_with_dem_dx['diagnosis'].isnull()), 
    'diagnosis'
] = diagnosis

In [ ]:
# 2315
target_rid = 2315
diagnosis = 2

mtb_merged_with_dem_dx.loc[
    (mtb_merged_with_dem_dx['RID'] == target_rid) & 
    (mtb_merged_with_dem_dx['diagnosis'].isnull()), 
    'diagnosis'
] = diagnosis

In [ ]:
# 2316
target_rid = 2316
diagnosis = 2

mtb_merged_with_dem_dx.loc[
    (mtb_merged_with_dem_dx['RID'] == target_rid) & 
    (mtb_merged_with_dem_dx['diagnosis'].isnull()), 
    'diagnosis'
] = diagnosis

In [ ]:
# 4174
target_rid = 4174
diagnosis = 1

mtb_merged_with_dem_dx.loc[
    (mtb_merged_with_dem_dx['RID'] == target_rid) & 
    (mtb_merged_with_dem_dx['diagnosis'].isnull()), 
    'diagnosis'
] = diagnosis

In [ ]:
# 4417
target_rid = 4417
diagnosis = 2

mtb_merged_with_dem_dx.loc[
    (mtb_merged_with_dem_dx['RID'] == target_rid) & 
    (mtb_merged_with_dem_dx['diagnosis'].isnull()), 
    'diagnosis'
] = diagnosis

In [ ]:
mtb_merged_with_dem_dx.isnull().sum()[mtb_merged_with_dem_dx.isnull().sum() > 0]

In [ ]:
pwd

In [ ]:
# not scaled or transformed
mtb_merged_with_dem_dx.to_pickle('../interim/clean_metabolites_and_diagnosis.pkl')

In [ ]:
mtb_merged_with_dem_dx['diagnosis'].isna().sum()

In [ ]:
mtb_merged_with_dem_dx[mtb_merged_with_dem_dx['RID'] == 2193]

In [ ]:
mtb_merged_with_dem_dx['RID'].nunique()

In [ ]:
MTB_PARTICIPANT_RIDS = list(mtb_merged_with_dem_dx['RID'].unique())

# Cognitive Tests

1. ADAS-Cog:
- Alzheimer's Disease Assessment Scale - Cognitive
- assesses cognitive function in areas like memory, language, and orientation, with higher scores indicating greater dysfunction, used as an index of global cognition in response to antidementia therapies
- two versions: score from 0 to 85 (TOTAL13) or 0-70 (TOTSCORE), with higher scores indicating greater cognitive dysfunction (it's used in clinical research to track changes over time, not for a simple diagnosis)

2. CDR:
- Clinical Dementia Rating
- evaluating memory, orientation, judgment and problem solving, community affairs, home and hobbies, and personal care
- ratings from 0 to 5, where 0 means no dementia, 0.5 indicates mild cognitive impairment (MCI), and higher scores represent increasing severity

3. GDS:
- Geriatric Depression Scale
- a score of 0-4 is normal, 5-8 is mild depression, 9-11 is moderate depression, and 12-15 is severe depression

4. MMSE:
-  Mini-Mental State Exam
- 30-point test of cognitive function,
- score of 25 or higher normal, 24 or lower cognitive impairment, with scores decreasing as severity increases (18-23 indicates mild impairment, while scores of 11-20 and 0-10 suggest moderate and severe impairment)

5. MOCA:
- Montreal Cognitive Assessment
- include short-term memory, executive function, attention, focus, and more
- normal score is 26 or higher out of a possible 30

6. NEUROBAT:
- Neuropsychological Battery
- 33 neuropsychological tests to assess skills in adults with neurocognitive dysfunction
- multiple assessments, including: Logical Memory, Rey Auditory Verbal Learning Test, Clock Drawing, Clock Copying, Category Fluency, Trail Making Test, Boston Naming Test, ANART, and Digit Span. At any given visit, only a subset of those tests are administered.

    a) Logical Memory - Immediate Recall: LMSTORY, LIMMTOTAL, LIMMEND

    b) Logical Memory - Delayed Recall: LDELBEGIN, LDELTOTAL, LDELCUE.

    c) Rey Auditory Verbal Learning Test: AVTOT1, AVERR1, AVTOT2, AVERR2, AVTOT3, AVERR3, AVTOT4, AVERR4, AVTOT5, AVERR5, AVTOT6, AVERR6, AVTOTB, AVERRB, AVENDED

    d) Rey Auditory Verbal Learning Test - Delayed: AVDELBEGAN, AVDEL30MIN, AVDELERR1, AVDELTOT, AVDELERR2
    Rey verbal learning test is a useful tool for differential diagnosis in the preclinical phase of Alzheimer's disease: comparison with MCI and normal aging. (Estevez-Gonzalez et al. 2003)

    e) Clock Drawing: CLOCKCIRC, CLOCKSYM, CLOCKNUM, CLOCKHAND, CLOCKTIME, CLOCKSCOR

    f) Clock Copying: COPYCIRC, COPYSYM, COPYNUM, COPYHAND, COPYTIME, COPYSCOR

    g) Category Fluency (Vegetables ADNI1 only): CATANIMSC, CATANPERS, CATANINTR, CATVEGESC, CATVGPERS, CATVGINTR

    h) Trail Making Test: TRAASCOR, TRAAERRCOM, TRAAERROM, TRABSCOR, TRABERRCOM, TRABERROM

    i) Boston Naming Test (Spanish version provided in ADNIGO/2): BNTND, BNTSPONT, BNTSTIM, BNTCSTIM, BNTPHON, BNTCPHON, BNTTOTAL

    j) American National Adult Reading Test (ANART): ANARTND, ANARTERR

    k) Digit Span (ADNI1 only): DSPANFOR, DSPANFLTH, DSPANBAC, DSPANBLTH

In [ ]:
# ADAS
ada = pd.read_csv('ADAS_15Oct2025.csv', low_memory=False)
# MOcA
moc = pd.read_csv('MOCA_16Oct2025.csv', low_memory=False)
# CDR
cdr = pd.read_csv('CDR_15Oct2025.csv', low_memory=False)
#GDSCALE
gds = pd.read_csv('GDSCALE_15Oct2025.csv', low_memory=False)
# MMSE
mms = pd.read_csv('MMSE_15Oct2025.csv', low_memory=False)
# NEUROBAT
nbt = pd.read_csv('NEUROBAT_16Oct2025.csv', low_memory=False)

## ADAS-Cog
Higher = worse   

Dementia Suggestion: A Global CDR of 1.0 or higher typically indicates dementia.   
    - 0: Healthy   
    - 0.5: "Very Mild" (Often the threshold for MCI)   
    - 1.0: Mild Dementia  
    - 2.0: Moderate Dementia  
    - 3.0: Severe Dementia   

ADAS-Cog is the "gold standard" for research sensitivity. The ADAS-Cog or CDR Sum of Boxes (CDR-SB) usually provides the "spread" needed to find that earlier signal.

In [ ]:
print(ada.shape)
print(ada.columns)
ada.head()

In [ ]:
ada[ada['TOTSCORE'] < 0]

In [ ]:
# Keep only RID of our subjects
ada = ada[ada['RID'].isin(MTB_PARTICIPANT_RIDS)]

In [ ]:
ada.isnull().sum()

In [ ]:
ada.dtypes

In [ ]:
ada.dropna(subset=['TOTSCORE'], inplace=True)

In [ ]:
ada['EXAMDATE'] = ada['VISDATE']

In [ ]:
ada[ada['EXAMDATE'].isna()]

In [ ]:
ada[ada['RID'] == 830]

In [ ]:
# Create a mask for the specific row(s)
mask = (ada['RID'] == 830) & (ada['EXAMDATE'].isna())

# Drop the rows matching that mask
ada = ada.drop(ada[mask].index)

In [ ]:
ada['EXAMDATE'] = pd.to_datetime(ada['EXAMDATE'])

In [ ]:
ada['VISCODE2'] = pd.Categorical(ada['VISCODE2'], categories=visit_order, ordered=True)

In [ ]:
ada[ada['VISCODE2'].isna()]

In [ ]:
ada.isnull().sum()

In [ ]:
ada = ada[['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'TOTSCORE']]

In [ ]:
ada.dtypes

In [ ]:
# Prepare to make VISCODE2 categorical
in_visit_order = ada[ada['VISCODE2'].isin(visit_order)]['VISCODE2'].unique()
not_in_visit_order = ada[~ada['VISCODE2'].isin(visit_order)]['VISCODE2'].unique()
print(in_visit_order)
print(not_in_visit_order)

In [ ]:
ada['VISCODE2'].cat.categories.tolist()

In [ ]:
# Over all phases?
ada['PHASE'].unique()

In [ ]:
ada.isnull().sum()

In [ ]:
dem_dx.columns

In [ ]:
subset = dem_dx[['PHASE', 'RID', 'VISCODE2', 'diagnosis', 'sex','birth_year']]

ada_dx_dem = pd.merge(
    ada,
    subset,
    on=['RID', 'VISCODE2', 'PHASE'],
    how='left'
)

In [ ]:
ada_dx_dem.isnull().sum()

In [ ]:
ada_dx_dem.shape

In [ ]:
ada_dx_dem.columns

In [ ]:
partially_missing_rids = (
    ada_dx_dem[ada_dx_dem['diagnosis']
    .isna()]['RID']
    .astype(int)
    .unique()
    .tolist()
)

In [ ]:
ada_dx_dem = fill_partially_missing_values(
    ada_dx_dem, 
    partially_missing_rids
    )

In [ ]:


cols_to_fill = ['sex', 'birth_year']
ada_dx_dem = fill_missing_with_known(
    ada_dx_dem, 
    cols_to_fill, 
    id_col='RID', 
    date_col='EXAMDATE'
    )

In [ ]:
ada_dx_dem.isnull().sum()

In [ ]:
ada_dx_dem[ada_dx_dem['diagnosis'].isna()]

In [ ]:
ada_dx_dem[ada_dx_dem['RID'] == 4198]

In [ ]:
ada_dx_dem = ada_dx_dem.fillna({'diagnosis': 1})

In [ ]:
# Write clean df to .pkl to preserve dates and categories
ada_dx_dem.to_pickle('../interim/clean_adas_dem_dx.pkl')

## Mini Mental State Examination (MMSE) - available through all phases

24-30 = healthy

20–23: Mild Impairment   
10–19: Moderate Impairment   
<10: Severe Impairment   

In [ ]:
# Keep only RID of our subjects
mms = mms[mms['RID'].isin(MTB_PARTICIPANT_RIDS)]

In [ ]:
print(mms.shape)
print(mms.dtypes)
mms.head()

In [ ]:
mms.isnull().sum()

In [ ]:
mms[mms['MMSCORE'].isna()].head()

In [ ]:
mms.dropna(subset=['MMSCORE'], inplace=True)

In [ ]:
# VISDATE dtype
mms['VISDATE'] = pd.to_datetime(mms['VISDATE'], errors='coerce')

# Order VISCODE2 column
mms = mms.copy()  # Ensure it’s a new copy
mms['VISCODE2'] = pd.Categorical(mms['VISCODE2'], categories=visit_order, ordered=True)

In [ ]:
print(mms.columns)

mms = mms[['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'MMDATE', 
            'MMYEAR', 'MMMONTH', 'MMDAY', 'MMSEASON', 'MMHOSPIT', 'MMFLOOR', 
            'MMCITY', 'MMAREA', 'MMSTATE', 'WORD1', 'WORD2', 
            'WORD3', 'MMD', 'MML', 'MMR', 'MMO', 'MMW', 'MMLTR1', 
            'MMLTR2', 'MMLTR3', 'MMLTR4', 'MMLTR5', 'MMLTR6', 'MMLTR7', 
            'WORLDSCORE', 'WORD1DL', 'WORD2DL', 'WORD3DL', 'MMWATCH', 
            'MMPENCIL', 'MMREPEAT', 'MMHAND', 'MMFOLD', 'MMONFLR', 'MMREAD', 
            'MMWRITE', 'MMDRAW', 'MMSCORE']]

In [ ]:
mms.shape

In [ ]:
mms.isna().sum()

In [ ]:
mms['EXAMDATE'] = mms['VISDATE']

In [ ]:
mms_clean = mms[['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'MMSCORE']]
mms_clean.isna().sum()

In [ ]:
mms_clean[mms_clean['EXAMDATE'].isna()]

In [ ]:
mms_clean[mms_clean['RID'] == 4555]

In [ ]:
mask = (mms_clean['RID'] == 4555) & (mms_clean['EXAMDATE'].isna())
mms_clean.loc[mask, 'EXAMDATE'] = pd.to_datetime('2015-03-20')

In [ ]:
mms_clean.dropna(subset=['EXAMDATE'], inplace=True)

In [ ]:
# Drop those who failed screening
mms_clean = mms_clean[mms_clean['VISCODE'] != 'f']
mms_clean.shape

In [ ]:
mms_dem_dx = pd.merge(
    mms_clean,
    subset,
    on=['PHASE', 'VISCODE2', 'RID'],
    how='left'
) 

In [ ]:
mms_dem_dx.head()

In [ ]:
mms_dem_dx.isnull().sum()

In [ ]:
partially_missing_rids = (
    mms_dem_dx[mms_dem_dx['diagnosis']
    .isna()]['RID']
    .astype(int)
    .unique()
    .tolist()
)

mms_dem_dx = fill_partially_missing_values(
    mms_dem_dx, 
    partially_missing_rids
    )

cols_to_fill = ['sex', 'birth_year']
mms_dem_dx = fill_missing_with_known(
    mms_dem_dx, 
    cols_to_fill, 
    id_col='RID', 
    date_col='EXAMDATE'
    )

In [ ]:
mms_dem_dx.dtypes

In [ ]:
mms_dem_dx = mms_dem_dx.sort_values(by=['RID', 'EXAMDATE'])
mms_dem_dx['diagnosis'] = mms_dem_dx.groupby('RID')['diagnosis'].ffill()

In [ ]:
mms_clean.to_pickle('../interim/clean_mmse_dem_dx.pkl')

## Clinical Dementia Rating (CDR)

0 = healthy

Global Clinical Dementia Rating (CDR) is a key 5-point scale (0–3) used to stage dementia severity:   
(0=none, 0.5=questionable, 1=mild, 2=moderate, 3=severe).

In [ ]:
%cd data/raw

In [ ]:
# CDR
cdr = pd.read_csv('CDR_15Oct2025.csv', low_memory=False)

In [ ]:
# Keep only participants who have metabolites
cdr = cdr[cdr['RID'].isin(MTB_PARTICIPANT_RIDS)]

In [ ]:
cdr.head()

In [ ]:
cdr['CDGLOBAL'].unique()

In [ ]:
# -1 is NaN
cdr['CDGLOBAL'] = cdr['CDGLOBAL'].replace(-1, np.nan)

In [ ]:
cdr.isnull().sum()

In [ ]:
cdr.columns[-8:]

In [ ]:
cdr.drop(columns=(list(cdr.columns[-8:]) + ['CDSOURCE', 'CDVERSION', 'SPID']), inplace=True)

In [ ]:
cdr[cdr['CDGLOBAL'].isna()].head()

In [ ]:
cdr[cdr['CDRSB'].isna()].shape

In [ ]:
cdr.dropna(subset=['CDGLOBAL'], inplace=True)

In [ ]:
# CDRSB=CDMEMORY+CDORIENT+CDJUDGE+CDCOMMUN+CDHOME+CDCARE
CDR_COMPONENTS = ['CDMEMORY', 'CDORIENT', 'CDJUDGE', 'CDCOMMUN', 'CDHOME', 'CDCARE']

cdrsb_computed = cdr[CDR_COMPONENTS].sum(axis=1)
missing_mask = cdr['CDRSB'].isna() & cdr[CDR_COMPONENTS].notna().all(axis=1) # only calculate if all 6 values are available
cdr.loc[missing_mask, 'CDRSB'] = cdrsb_computed[missing_mask]


In [ ]:
%pwd

In [ ]:
cdr[cdr['CDRSB'] <0 ]

In [ ]:
cdr = cdr[['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'CDRSB', 'CDGLOBAL']]

In [ ]:
cdr.head()

In [ ]:
cdr.isnull().sum()

In [ ]:
cdr[cdr['VISCODE2'].isna()]

In [ ]:
cdr[cdr['RID'] == 2119]

In [ ]:
# how long doe eit take from init to first year? more like 6m or 12 m?
cdr[(cdr['VISCODE'] == 'init') & (cdr['PHASE'] == 'ADNI3')].head(10)

In [ ]:
cdr[cdr['RID'] == 4214] # Takes 12 m

In [ ]:
# Are these dates available in registry?
reg[reg['RID'].isin([2119, 4414])]

In [ ]:
cdr.loc[(cdr['RID'] == 2119) & (cdr['VISCODE2'].isna()), 'VISCODE2'] = 'm90' # 12 m from m78 at init
cdr.loc[(cdr['RID'] == 4414) & (cdr['VISCODE2'].isna()), 'VISCODE2'] = 'm78'

In [ ]:
cdr.isnull().sum()

In [ ]:
cdr[cdr['VISDATE'].isna()]

In [ ]:
reg[reg['RID'].isin([830])]

In [ ]:
cdr.loc[(cdr['RID'] == 830) & (cdr['VISDATE'].isna()), 'VISDATE'] = '2014-07-06' # from reg 'USERDATE

In [ ]:
cdr[cdr['RID'] == 4414]

In [ ]:
cdr.loc[(cdr['RID'] == 2119) & (cdr['VISDATE'].isna()), 'VISDATE'] = '2018-06-13'
cdr.loc[(cdr['RID'] == 4414) & (cdr['VISDATE'].isna()), 'VISDATE'] = '2018-10-16' # deducted from reg['USERDATE']

In [ ]:
cdr['VISDATE'].dtype

In [ ]:
cdr['VISDATE'] = pd.to_datetime(cdr['VISDATE']).dt.date

In [ ]:
subset.head()

In [ ]:
cdr.head()

In [ ]:
cdr_dem_dx = pd.merge(
    cdr,
    subset, # demographic data
    on = ['PHASE', 'VISCODE2', 'RID'],
    how='left'
)

In [ ]:
cdr_dem_dx.head()

In [ ]:
partially_missing_rids = (
    cdr_dem_dx[cdr_dem_dx['diagnosis']
    .isna()]['RID']
    .astype(int)
    .unique()
    .tolist()
)

cdr_dem_dx = fill_partially_missing_values(
    cdr_dem_dx, 
    partially_missing_rids
    )

cols_to_fill = ['sex', 'birth_year']
cdr_dem_dx = fill_missing_with_known(
    cdr_dem_dx, 
    cols_to_fill, 
    id_col='RID', 
    date_col='VISDATE'
    )

In [ ]:
cdr_dem_dx.isnull().sum()

In [ ]:
cdr_dem_dx['VISDATE'] = pd.to_datetime(cdr_dem_dx['VISDATE'], errors='coerce')

In [ ]:
cdr_dem_dx = cdr_dem_dx.sort_values(by=['RID', 'VISDATE'])
cdr_dem_dx['diagnosis'] = cdr_dem_dx.groupby('RID')['diagnosis'].ffill()

In [ ]:
cdr_dem_dx.isnull().sum()

In [ ]:
cdr_dem_dx.to_pickle('../interim/clean_cdr_dem_dx.pkl')

## GDS

The **GDS (Geriatric Depression Scale) dataset** records self-reported depressive symptom scores across ADNI visits. Depression is a known risk factor and early indicator in AD (inflammation hypothesis); including GDS allows us to control for mood-related confounders when modelling cognitive decline and metabolite associations.

In [ ]:
gds.head()

In [ ]:
gds.isnull().sum()

In [ ]:
gds.columns.to_list()

In [ ]:
cols_to_drop = [
 'SOURCE',
 'ID',
 'SITEID',
 'USERDATE',
 'USERDATE2',
 'DD_CRF_VERSION_LABEL',
 'LANGUAGE_CODE',
 'HAS_QC_ERROR',
 'update_stamp'
 ]

gds.drop(columns=cols_to_drop, inplace=True)

print(gds.dtypes)
gds.dropna(subset=['GDTOTAL'], inplace=True)
gds.isnull().sum()

In [ ]:
# gds.to_pickle('clean_gds.pkl')

## Montreal Cognitive Assessment (MoCA)

The **MoCA dataset** captures scores from the Montreal Cognitive Assessment, a rapid screening tool sensitive to mild cognitive impairment (scores below 26 suggest MCI, max 30). MoCA is available across all ADNI phases and complements MMSE by being more sensitive to early executive function and memory deficits.

In [ ]:
# Montreal Cognitive Assessment (MoCA) available through all phases
moc.columns

In [ ]:
print(moc.shape)
moc = moc[['PHASE','RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'TRAILS',
       'CUBE', 'CLOCKCON', 'CLOCKNO', 'CLOCKHAN', 'LION', 'RHINO', 'CAMEL',
       'IMMT1W1', 'IMMT1W2', 'IMMT1W3', 'IMMT1W4', 'IMMT1W5', 'IMMT2W1',
       'IMMT2W2', 'IMMT2W3', 'IMMT2W4', 'IMMT2W5', 'DIGFOR', 'DIGBACK',
       'LETTERS', 'SERIAL1', 'SERIAL2', 'SERIAL3', 'SERIAL4', 'SERIAL5',
       'REPEAT1', 'REPEAT2', 'FFLUENCY', 'ABSTRAN', 'ABSMEAS', 'DELW1',
       'DELW2', 'DELW3', 'DELW4', 'DELW5', 'DATE', 'MONTH', 'YEAR', 'DAY',
       'PLACE', 'CITY', 'MOCA']]
moc.head()

In [ ]:
moc.isnull().sum()

In [ ]:
moc[moc['VISCODE2'].isna()]

In [ ]:
print(moc[moc['RID'] == 6082][['PHASE', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE']])
print(moc[moc['RID'] == 2068][['PHASE', 'RID', 'VISCODE', 'VISCODE2','VISDATE']])
print(moc[moc['RID'] == 5273][['PHASE', 'RID', 'VISCODE', 'VISCODE2','VISDATE']])
print(dem[dem['RID'] == 5273][['PHASE', 'RID', 'VISCODE', 'VISDATE']])

In [ ]:
# Manually fix missing VISCODE2 by checking other data per RID
moc.loc[(moc['RID'] == 6082) & (moc['VISCODE2'].isna()), 'VISCODE2'] = 'm12'
moc.loc[(moc['RID'] == 2068) & (moc['VISCODE2'].isna()), 'VISCODE2'] = 'm96'
moc.loc[(moc['RID'] == 5273) & (moc['VISCODE2'].isna()), 'VISCODE2'] = 'm90'
moc.loc[(moc['RID'] == 6970) & (moc['VISCODE2'].isna()), 'VISCODE2'] = 'm42'
moc.loc[(moc['RID'] == 10529) & (moc['VISCODE2'].isna()), 'VISCODE2'] = 'bl'
moc.loc[(moc['RID'] == 10005) & (moc['VISCODE2'].isna()), 'VISCODE2'] = 'm24'

In [ ]:
moc.isnull().sum(0)

In [ ]:
moc[moc['TRAILS'].isna()].head()

It is impossible to say if the NAN are supposed to be 0.0 or truly NaN, or this are just answers that were too easy for the candidateand they were skipped?

In [ ]:
print(moc.shape)
# Drop rows where all values are NaN in columns from 5 to the second-to-last column
moc.dropna(subset=moc.columns[5:-1], how='all', inplace=True)
print(moc.shape)

In [ ]:
moc_clean = moc.dropna(subset=moc.columns[5:-1], how='any')
moc_clean.shape

In [ ]:
moc_clean.tail()

In [ ]:
moc_clean.isnull().sum(0)

In [ ]:
# VISDATE dtype
moc_clean['VISDATE'] = pd.to_datetime(moc_clean['VISDATE'], errors='coerce')

In [ ]:
moc_clean.head()

In [ ]:
# Write clean df to .pkl to preserve dates and categories
moc_clean.to_pickle('../interim/clean_moca.pkl')

## PET Amyloid Tabular Data

The **PET Amyloid dataset** contains region-level SUVRs and volumes from amyloid PET scans covering cortical and subcortical regions. Each row is one participant–visit combination. Amyloid PET is considered the gold-standard biomarker for detecting amyloid plaques, a hallmark of preclinical and clinical Alzheimer's disease.

In [ ]:
amy_pet = load_and_preprocess_pet_data()
amy_pet.head()

In [ ]:
amy_pet.columns.to_list()

In [ ]:
amy_pet = amy_pet[amy_pet['RID'].isin(MTB_PARTICIPANT_RIDS)]

In [ ]:
amy_pet = amy_pet[[
    'RID',
    'VISCODE',
    'VISCODE2',
    'SCANDATE',
    'TRACER',
    'ACCUMBENS_AREA_SUVR',
    'ACCUMBENS_AREA_VOLUME',
    'AMYGDALA_SUVR',
    'AMYGDALA_VOLUME',
    'BRAINSTEM_SUVR',
    'BRAINSTEM_VOLUME',
    'CAUDATE_SUVR',
    'CAUDATE_VOLUME',
    'CC_ANTERIOR_SUVR',
    'CC_ANTERIOR_VOLUME',
    'CC_CENTRAL_SUVR',
    'CC_CENTRAL_VOLUME',
    'CC_MID_ANTERIOR_SUVR',
    'CC_MID_ANTERIOR_VOLUME',
    'CC_MID_POSTERIOR_SUVR',
    'CC_MID_POSTERIOR_VOLUME',
    'CC_POSTERIOR_SUVR',
    'CC_POSTERIOR_VOLUME',
    'CEREBELLUM_CORTEX_SUVR',
    'CEREBELLUM_CORTEX_VOLUME',
    'CEREBELLUM_WHITE_MATTER_SUVR',
    'CEREBELLUM_WHITE_MATTER_VOLUME',
    'CEREBRAL_WHITE_MATTER_SUVR',
    'CEREBRAL_WHITE_MATTER_VOLUME',
    'CHOROID_PLEXUS_SUVR',
    'CHOROID_PLEXUS_VOLUME',
    'CSF_SUVR',
    'CSF_VOLUME',
    'CTX_BANKSSTS_SUVR',
    'CTX_BANKSSTS_VOLUME',
    'CTX_CAUDALANTERIORCINGULATE_SUVR',
    'CTX_CAUDALANTERIORCINGULATE_VOLUME',
    'CTX_CAUDALMIDDLEFRONTAL_SUVR',
    'CTX_CAUDALMIDDLEFRONTAL_VOLUME',
    'CTX_CUNEUS_SUVR',
    'CTX_CUNEUS_VOLUME',
    'CTX_ENTORHINAL_SUVR',
    'CTX_ENTORHINAL_VOLUME',
    'CTX_FRONTALPOLE_SUVR',
    'CTX_FRONTALPOLE_VOLUME',
    'CTX_FUSIFORM_SUVR',
    'CTX_FUSIFORM_VOLUME',
    'CTX_INFERIORPARIETAL_SUVR',
    'CTX_INFERIORPARIETAL_VOLUME',
    'CTX_INFERIORTEMPORAL_SUVR',
    'CTX_INFERIORTEMPORAL_VOLUME',
    'CTX_INSULA_SUVR',
    'CTX_INSULA_VOLUME',
    'CTX_ISTHMUSCINGULATE_SUVR',
    'CTX_ISTHMUSCINGULATE_VOLUME',
    'CTX_LATERALOCCIPITAL_SUVR',
    'CTX_LATERALOCCIPITAL_VOLUME',
    'CTX_LATERALORBITOFRONTAL_SUVR',
    'CTX_LATERALORBITOFRONTAL_VOLUME',
    'CTX_LH_BANKSSTS_SUVR',
    'CTX_LH_BANKSSTS_VOLUME',
    'CTX_LH_CAUDALANTERIORCINGULATE_SUVR',
    'CTX_LH_CAUDALANTERIORCINGULATE_VOLUME',
    'CTX_LH_CAUDALMIDDLEFRONTAL_SUVR',
    'CTX_LH_CAUDALMIDDLEFRONTAL_VOLUME',
    'CTX_LH_CUNEUS_SUVR',
    'CTX_LH_CUNEUS_VOLUME',
    'CTX_LH_ENTORHINAL_SUVR',
    'CTX_LH_ENTORHINAL_VOLUME',
    'CTX_LH_FRONTALPOLE_SUVR',
    'CTX_LH_FRONTALPOLE_VOLUME',
    'CTX_LH_FUSIFORM_SUVR',
    'CTX_LH_FUSIFORM_VOLUME',
    'CTX_LH_INFERIORPARIETAL_SUVR',
    'CTX_LH_INFERIORPARIETAL_VOLUME',
    'CTX_LH_INFERIORTEMPORAL_SUVR',
    'CTX_LH_INFERIORTEMPORAL_VOLUME',
    'CTX_LH_INSULA_SUVR',
    'CTX_LH_INSULA_VOLUME',
    'CTX_LH_ISTHMUSCINGULATE_SUVR',
    'CTX_LH_ISTHMUSCINGULATE_VOLUME',
    'CTX_LH_LATERALOCCIPITAL_SUVR',
    'CTX_LH_LATERALOCCIPITAL_VOLUME',
    'CTX_LH_LATERALORBITOFRONTAL_SUVR',
    'CTX_LH_LATERALORBITOFRONTAL_VOLUME',
    'CTX_LH_LINGUAL_SUVR',
    'CTX_LH_LINGUAL_VOLUME',
    'CTX_LH_MEDIALORBITOFRONTAL_SUVR',
    'CTX_LH_MEDIALORBITOFRONTAL_VOLUME',
    'CTX_LH_MIDDLETEMPORAL_SUVR',
    'CTX_LH_MIDDLETEMPORAL_VOLUME',
    'CTX_LH_PARACENTRAL_SUVR',
    'CTX_LH_PARACENTRAL_VOLUME',
    'CTX_LH_PARAHIPPOCAMPAL_SUVR',
    'CTX_LH_PARAHIPPOCAMPAL_VOLUME',
    'CTX_LH_PARSOPERCULARIS_SUVR',
    'CTX_LH_PARSOPERCULARIS_VOLUME',
    'CTX_LH_PARSORBITALIS_SUVR',
    'CTX_LH_PARSORBITALIS_VOLUME',
    'CTX_LH_PARSTRIANGULARIS_SUVR',
    'CTX_LH_PARSTRIANGULARIS_VOLUME',
    'CTX_LH_PERICALCARINE_SUVR',
    'CTX_LH_PERICALCARINE_VOLUME',
    'CTX_LH_POSTCENTRAL_SUVR',
    'CTX_LH_POSTCENTRAL_VOLUME',
    'CTX_LH_POSTERIORCINGULATE_SUVR',
    'CTX_LH_POSTERIORCINGULATE_VOLUME',
    'CTX_LH_PRECENTRAL_SUVR',
    'CTX_LH_PRECENTRAL_VOLUME',
    'CTX_LH_PRECUNEUS_SUVR',
    'CTX_LH_PRECUNEUS_VOLUME',
    'CTX_LH_ROSTRALANTERIORCINGULATE_SUVR',
    'CTX_LH_ROSTRALANTERIORCINGULATE_VOLUME',
    'CTX_LH_ROSTRALMIDDLEFRONTAL_SUVR',
    'CTX_LH_ROSTRALMIDDLEFRONTAL_VOLUME',
    'CTX_LH_SUPERIORFRONTAL_SUVR',
    'CTX_LH_SUPERIORFRONTAL_VOLUME',
    'CTX_LH_SUPERIORPARIETAL_SUVR',
    'CTX_LH_SUPERIORPARIETAL_VOLUME',
    'CTX_LH_SUPERIORTEMPORAL_SUVR',
    'CTX_LH_SUPERIORTEMPORAL_VOLUME',
    'CTX_LH_SUPRAMARGINAL_SUVR',
    'CTX_LH_SUPRAMARGINAL_VOLUME',
    'CTX_LH_TEMPORALPOLE_SUVR',
    'CTX_LH_TEMPORALPOLE_VOLUME',
    'CTX_LH_TRANSVERSETEMPORAL_SUVR',
    'CTX_LH_TRANSVERSETEMPORAL_VOLUME',
    'CTX_LINGUAL_SUVR',
    'CTX_LINGUAL_VOLUME',
    'CTX_MEDIALORBITOFRONTAL_SUVR',
    'CTX_MEDIALORBITOFRONTAL_VOLUME',
    'CTX_MIDDLETEMPORAL_SUVR',
    'CTX_MIDDLETEMPORAL_VOLUME',
    'CTX_PARACENTRAL_SUVR',
    'CTX_PARACENTRAL_VOLUME',
    'CTX_PARAHIPPOCAMPAL_SUVR',
    'CTX_PARAHIPPOCAMPAL_VOLUME',
    'CTX_PARSOPERCULARIS_SUVR',
    'CTX_PARSOPERCULARIS_VOLUME',
    'CTX_PARSORBITALIS_SUVR',
    'CTX_PARSORBITALIS_VOLUME',
    'CTX_PARSTRIANGULARIS_SUVR',
    'CTX_PARSTRIANGULARIS_VOLUME',
    'CTX_PERICALCARINE_SUVR',
    'CTX_PERICALCARINE_VOLUME',
    'CTX_POSTCENTRAL_SUVR',
    'CTX_POSTCENTRAL_VOLUME',
    'CTX_POSTERIORCINGULATE_SUVR',
    'CTX_POSTERIORCINGULATE_VOLUME',
    'CTX_PRECENTRAL_SUVR',
    'CTX_PRECENTRAL_VOLUME',
    'CTX_PRECUNEUS_SUVR',
    'CTX_PRECUNEUS_VOLUME',
    'CTX_RH_BANKSSTS_SUVR',
    'CTX_RH_BANKSSTS_VOLUME',
    'CTX_RH_CAUDALANTERIORCINGULATE_SUVR',
    'CTX_RH_CAUDALANTERIORCINGULATE_VOLUME',
    'CTX_RH_CAUDALMIDDLEFRONTAL_SUVR',
    'CTX_RH_CAUDALMIDDLEFRONTAL_VOLUME',
    'CTX_RH_CUNEUS_SUVR',
    'CTX_RH_CUNEUS_VOLUME',
    'CTX_RH_ENTORHINAL_SUVR',
    'CTX_RH_ENTORHINAL_VOLUME',
    'CTX_RH_FRONTALPOLE_SUVR',
    'CTX_RH_FRONTALPOLE_VOLUME',
    'CTX_RH_FUSIFORM_SUVR',
    'CTX_RH_FUSIFORM_VOLUME',
    'CTX_RH_INFERIORPARIETAL_SUVR',
    'CTX_RH_INFERIORPARIETAL_VOLUME',
    'CTX_RH_INFERIORTEMPORAL_SUVR',
    'CTX_RH_INFERIORTEMPORAL_VOLUME',
    'CTX_RH_INSULA_SUVR',
    'CTX_RH_INSULA_VOLUME',
    'CTX_RH_ISTHMUSCINGULATE_SUVR',
    'CTX_RH_ISTHMUSCINGULATE_VOLUME',
    'CTX_RH_LATERALOCCIPITAL_SUVR',
    'CTX_RH_LATERALOCCIPITAL_VOLUME',
    'CTX_RH_LATERALORBITOFRONTAL_SUVR',
    'CTX_RH_LATERALORBITOFRONTAL_VOLUME',
    'CTX_RH_LINGUAL_SUVR',
    'CTX_RH_LINGUAL_VOLUME',
    'CTX_RH_MEDIALORBITOFRONTAL_SUVR',
    'CTX_RH_MEDIALORBITOFRONTAL_VOLUME',
    'CTX_RH_MIDDLETEMPORAL_SUVR',
    'CTX_RH_MIDDLETEMPORAL_VOLUME',
    'CTX_RH_PARACENTRAL_SUVR',
    'CTX_RH_PARACENTRAL_VOLUME',
    'CTX_RH_PARAHIPPOCAMPAL_SUVR',
    'CTX_RH_PARAHIPPOCAMPAL_VOLUME',
    'CTX_RH_PARSOPERCULARIS_SUVR',
    'CTX_RH_PARSOPERCULARIS_VOLUME',
    'CTX_RH_PARSORBITALIS_SUVR',
    'CTX_RH_PARSORBITALIS_VOLUME',
    'CTX_RH_PARSTRIANGULARIS_SUVR',
    'CTX_RH_PARSTRIANGULARIS_VOLUME',
    'CTX_RH_PERICALCARINE_SUVR',
    'CTX_RH_PERICALCARINE_VOLUME',
    'CTX_RH_POSTCENTRAL_SUVR',
    'CTX_RH_POSTCENTRAL_VOLUME',
    'CTX_RH_POSTERIORCINGULATE_SUVR',
    'CTX_RH_POSTERIORCINGULATE_VOLUME',
    'CTX_RH_PRECENTRAL_SUVR',
    'CTX_RH_PRECENTRAL_VOLUME',
    'CTX_RH_PRECUNEUS_SUVR',
    'CTX_RH_PRECUNEUS_VOLUME',
    'CTX_RH_ROSTRALANTERIORCINGULATE_SUVR',
    'CTX_RH_ROSTRALANTERIORCINGULATE_VOLUME',
    'CTX_RH_ROSTRALMIDDLEFRONTAL_SUVR',
    'CTX_RH_ROSTRALMIDDLEFRONTAL_VOLUME',
    'CTX_RH_SUPERIORFRONTAL_SUVR',
    'CTX_RH_SUPERIORFRONTAL_VOLUME',
    'CTX_RH_SUPERIORPARIETAL_SUVR',
    'CTX_RH_SUPERIORPARIETAL_VOLUME',
    'CTX_RH_SUPERIORTEMPORAL_SUVR',
    'CTX_RH_SUPERIORTEMPORAL_VOLUME',
    'CTX_RH_SUPRAMARGINAL_SUVR',
    'CTX_RH_SUPRAMARGINAL_VOLUME',
    'CTX_RH_TEMPORALPOLE_SUVR',
    'CTX_RH_TEMPORALPOLE_VOLUME',
    'CTX_RH_TRANSVERSETEMPORAL_SUVR',
    'CTX_RH_TRANSVERSETEMPORAL_VOLUME',
    'CTX_ROSTRALANTERIORCINGULATE_SUVR',
    'CTX_ROSTRALANTERIORCINGULATE_VOLUME',
    'CTX_ROSTRALMIDDLEFRONTAL_SUVR',
    'CTX_ROSTRALMIDDLEFRONTAL_VOLUME',
    'CTX_SUPERIORFRONTAL_SUVR',
    'CTX_SUPERIORFRONTAL_VOLUME',
    'CTX_SUPERIORPARIETAL_SUVR',
    'CTX_SUPERIORPARIETAL_VOLUME',
    'CTX_SUPERIORTEMPORAL_SUVR',
    'CTX_SUPERIORTEMPORAL_VOLUME',
    'CTX_SUPRAMARGINAL_SUVR',
    'CTX_SUPRAMARGINAL_VOLUME',
    'CTX_TEMPORALPOLE_SUVR',
    'CTX_TEMPORALPOLE_VOLUME',
    'CTX_TRANSVERSETEMPORAL_SUVR',
    'CTX_TRANSVERSETEMPORAL_VOLUME',
    'ERODED_SUBCORTICALWM_SUVR',
    'ERODED_SUBCORTICALWM_VOLUME',
    'HIPPOCAMPUS_SUVR',
    'HIPPOCAMPUS_VOLUME',
    'INF_LAT_VENT_SUVR',
    'INF_LAT_VENT_VOLUME',
    # 'INFERIORCEREBELLUM_SUVR',
    # 'INFERIORCEREBELLUM_VOLUME',
    'LATERAL_VENTRICLE_SUVR',
    'LATERAL_VENTRICLE_VOLUME',
    'LEFT_ACCUMBENS_AREA_SUVR',
    'LEFT_ACCUMBENS_AREA_VOLUME',
    'LEFT_AMYGDALA_SUVR',
    'LEFT_AMYGDALA_VOLUME',
    'LEFT_CAUDATE_SUVR',
    'LEFT_CAUDATE_VOLUME',
    'LEFT_CEREBELLUM_CORTEX_SUVR',
    'LEFT_CEREBELLUM_CORTEX_VOLUME',
    'LEFT_CEREBELLUM_WHITE_MATTER_SUVR',
    'LEFT_CEREBELLUM_WHITE_MATTER_VOLUME',
    'LEFT_CEREBRAL_WHITE_MATTER_SUVR',
    'LEFT_CEREBRAL_WHITE_MATTER_VOLUME',
    'LEFT_CHOROID_PLEXUS_SUVR',
    'LEFT_CHOROID_PLEXUS_VOLUME',
    'LEFT_HIPPOCAMPUS_SUVR',
    'LEFT_HIPPOCAMPUS_VOLUME',
    'LEFT_INF_LAT_VENT_SUVR',
    'LEFT_INF_LAT_VENT_VOLUME',
    'LEFT_LATERAL_VENTRICLE_SUVR',
    'LEFT_LATERAL_VENTRICLE_VOLUME',
    'LEFT_PALLIDUM_SUVR',
    'LEFT_PALLIDUM_VOLUME',
    'LEFT_PUTAMEN_SUVR',
    'LEFT_PUTAMEN_VOLUME',
    'LEFT_THALAMUS_PROPER_SUVR',
    'LEFT_THALAMUS_PROPER_VOLUME',
    'LEFT_VENTRALDC_SUVR',
    'LEFT_VENTRALDC_VOLUME',
    'LEFT_VESSEL_SUVR',
    'LEFT_VESSEL_VOLUME',
    # 'META_TEMPORAL_SUVR',
    # 'META_TEMPORAL_VOLUME',
    'NON_WM_HYPOINTENSITIES_SUVR',
    'NON_WM_HYPOINTENSITIES_VOLUME',
    'OPTIC_CHIASM_SUVR',
    'OPTIC_CHIASM_VOLUME',
    'PALLIDUM_SUVR',
    'PALLIDUM_VOLUME',
    'PUTAMEN_SUVR',
    'PUTAMEN_VOLUME',
    'RIGHT_ACCUMBENS_AREA_SUVR',
    'RIGHT_ACCUMBENS_AREA_VOLUME',
    'RIGHT_AMYGDALA_SUVR',
    'RIGHT_AMYGDALA_VOLUME',
    'RIGHT_CAUDATE_SUVR',
    'RIGHT_CAUDATE_VOLUME',
    'RIGHT_CEREBELLUM_CORTEX_SUVR',
    'RIGHT_CEREBELLUM_CORTEX_VOLUME',
    'RIGHT_CEREBELLUM_WHITE_MATTER_SUVR',
    'RIGHT_CEREBELLUM_WHITE_MATTER_VOLUME',
    'RIGHT_CEREBRAL_WHITE_MATTER_SUVR',
    'RIGHT_CEREBRAL_WHITE_MATTER_VOLUME',
    'RIGHT_CHOROID_PLEXUS_SUVR',
    'RIGHT_CHOROID_PLEXUS_VOLUME',
    'RIGHT_HIPPOCAMPUS_SUVR',
    'RIGHT_HIPPOCAMPUS_VOLUME',
    'RIGHT_INF_LAT_VENT_SUVR',
    'RIGHT_INF_LAT_VENT_VOLUME',
    'RIGHT_LATERAL_VENTRICLE_SUVR',
    'RIGHT_LATERAL_VENTRICLE_VOLUME',
    'RIGHT_PALLIDUM_SUVR',
    'RIGHT_PALLIDUM_VOLUME',
    'RIGHT_PUTAMEN_SUVR',
    'RIGHT_PUTAMEN_VOLUME',
    'RIGHT_THALAMUS_PROPER_SUVR',
    'RIGHT_THALAMUS_PROPER_VOLUME',
    'RIGHT_VENTRALDC_SUVR',
    'RIGHT_VENTRALDC_VOLUME',
    'RIGHT_VESSEL_SUVR',
    'RIGHT_VESSEL_VOLUME',
    'THALAMUS_PROPER_SUVR',
    'THALAMUS_PROPER_VOLUME',
    'VENTRALDC_SUVR',
    'VENTRALDC_VOLUME',
    'VENTRICLE_3RD_SUVR',
    'VENTRICLE_3RD_VOLUME',
    'VENTRICLE_4TH_SUVR',
    'VENTRICLE_4TH_VOLUME',
    'VENTRICLE_5TH_SUVR',
    'VENTRICLE_5TH_VOLUME',
    'VESSEL_SUVR',
    'VESSEL_VOLUME',
    'WM_HYPOINTENSITIES_SUVR',
    'WM_HYPOINTENSITIES_VOLUME'
]]

In [ ]:
amy_pet['VISCODE2'].unique()

In [ ]:
amy_pet.isnull().sum()

In [ ]:
# Num. of missing values, num. of rows
amy_pet.isnull().sum(axis=1).value_counts()

In [ ]:
amy_pet.shape

In [ ]:
amy_pet.drop(columns=[
    'VENTRICLE_5TH_SUVR', 
    'VENTRICLE_5TH_VOLUME',
    'NON_WM_HYPOINTENSITIES_SUVR',
    'NON_WM_HYPOINTENSITIES_VOLUME'
    ], inplace=True)
# No NA dataframe

In [ ]:
# rows missing most of PET data (90%), we drop them
amy_pet = amy_pet[amy_pet.isnull().sum(axis=1) < 300]

In [ ]:
amy_pet.shape

In [ ]:
amy_pet.isnull().sum()

With very few missing values, we can impute those. IterativeImputer (MICE) is robust and suited to clinical datasets where features (different brain regions) are related to one another.

In [ ]:
amy_pet.dtypes

In [ ]:
dem_dx.head()

In [ ]:
dem_dx.columns

In [ ]:
def add_diagnosis(pet_df, dx_df):
    pet_df = pet_df.merge(
        dx_df[['RID', 'VISCODE2', 'diagnosis', 'sex', 'birth_year']],
        on=['RID', 'VISCODE2'],
        how='left'
    )
    pet_df = pet_df.sort_values(by=['RID', 'VISCODE2'])
    pet_df['diagnosis'] = pet_df.groupby('RID')['diagnosis'].ffill()
    return pet_df

amy_pet = add_diagnosis(amy_pet, dem_dx)

In [ ]:
amy_pet.isnull().sum()

In [ ]:
missing_dx = amy_pet[amy_pet['diagnosis'].isna()]
missing_dx

In [ ]:
amy_pet['birth_year'] = amy_pet.groupby('RID')['birth_year'].bfill()
amy_pet['sex'] = amy_pet.groupby('RID')['sex'].bfill()

In [ ]:
amy_pet.isnull().sum()

In [ ]:
target_rids = amy_pet[amy_pet['sex'].isna()]['RID'].unique()

In [ ]:
# Filter dem_dx for these RIDs and the specific columns
check_df = dem_dx.loc[
    dem_dx['RID'].isin(target_rids), 
    ['RID', 'sex', 'birth_year']
].drop_duplicates()

print(check_df)

In [ ]:
sex_map = dem_dx.drop_duplicates('RID').set_index('RID')['sex']
birth_year_map = dem_dx.drop_duplicates('RID').set_index('RID')['birth_year']

# Fill NaNs in amy_pet using the mapping
amy_pet['sex'] = amy_pet['sex'].fillna(amy_pet['RID'].map(sex_map))
amy_pet['birth_year'] = amy_pet['birth_year'].fillna(amy_pet['RID'].map(birth_year_map))

In [ ]:
counts = amy_pet.nunique()
constant_columns = counts[counts == 1].index.tolist()
constant_columns

In [ ]:
amy_pet['VISCODE2'] = pd.Categorical(amy_pet['VISCODE2'], categories=visit_order, ordered=True)

In [ ]:
# amy_pet.drop(columns=['INFERIORCEREBELLUM_SUVR'], inplace=True)

'INFERIORCEREBELLUM_SUVR' only has one value = 1. If a column has only one value (all 1.0), the regression math fails and we cannot impute.    


The Inferior Cerebellum (Cerebellum Gray Matter) is used as the Reference Region. When researchers "normalize" PET scans, they divide every brain region's value by the value of the reference region. This results in the reference region itself having a value of exactly 1.0 for every single subject.

In [ ]:
amy_pet['birth_year'] = amy_pet['birth_year'].astype('int')
amy_pet.dtypes

We are not interested in blood vessels, but in amyloid deposits, so we can drop vessel columns.
()'_SUVR', Standardized Uptake Value Ratio, measures how much of a radioactive tracer (for amyloid or tau) is taken up in a brain region relative to a reference region.

In [ ]:
amy_pet.columns.tolist()

In [ ]:
amy_pet.drop(columns=['LEFT_VESSEL_SUVR', 'LEFT_VESSEL_VOLUME', 'RIGHT_VESSEL_SUVR', 'RIGHT_VESSEL_VOLUME'], inplace=True)
amy_pet = amy_pet.dropna()

In [ ]:
amy_pet[amy_pet.isna().any(axis=1)]

In [ ]:
amy_pet.to_pickle('../interim/clean_amy_pet_dem_dx.pkl')

## PET Tau Tabular Data

The **PET Tau dataset** contains SUVRs and volumes from PET scans covering 68 cortical and subcortical regions. Tau PET quantifies neurofibrillary tangle burden, a downstream marker of neurodegeneration that closely tracks cognitive decline and disease severity in Alzheimer's disease.

In [ ]:
tau_pet = load_and_preprocess_pet_data(filepath='PET_TAU_analysis.csv')
tau_pet.head()

In [ ]:
tau_pet = tau_pet[tau_pet['RID'].isin(MTB_PARTICIPANT_RIDS)]

In [ ]:
tau_pet.columns.to_list()

In [ ]:
start_cols = ['RID', 'VISCODE', 'VISCODE2', 'SCANDATE', 'TRACER']
remaining_cols = [col for col in tau_pet.columns if col not in start_cols]
tau_pet = tau_pet[start_cols + remaining_cols]

In [ ]:
inspect_df(tau_pet)

Tau PET has too few observations, probably even less such that have a serum gut metabolite history.

In [ ]:
tau_pet.drop(columns=[
    'NON_WM_HYPOINTENSITIES_SUVR',
    'NON_WM_HYPOINTENSITIES_VOLUME',
    'VENTRICLE_5TH_SUVR',
    'VENTRICLE_5TH_VOLUME'
    ], 
    inplace=True)

tau_pet.dropna(inplace=True)

In [ ]:
tau_pet = add_diagnosis(tau_pet, dem_dx)

In [ ]:
tau_pet.isnull().sum()

In [ ]:
tau_pet[tau_pet['diagnosis'].isna()]

In [ ]:
def fill_missing(target_df, source_df, columns):
    # 1. Get the latest record for each RID from the source
    # We use .copy() to ensure source_latest is its own object
    source_latest = source_df.sort_values(['RID', 'EXAMDATE']).groupby('RID').last()

    for col in columns:
        # 2. Ensure the column exists in target_df (avoids the warning on new columns)
        if col not in target_df.columns:
            target_df.loc[:, col] = pd.NA
        
        # 3. Use .loc[:, col] to fill values safely
        mapping = target_df['RID'].map(source_latest[col])
        target_df.loc[:, col] = target_df[col].fillna(mapping)
    
    return target_df

In [ ]:
cols = ['diagnosis', 'sex', 'birth_year']

# Update tau_pet
tau_pet = fill_missing(tau_pet, dem_dx, cols)

# Update amy_pet
amy_pet = fill_missing(amy_pet, dem_dx, cols)

In [ ]:
tau_pet.shape